# Revisiting the Strategies in Federated Learning

As mentioned in the previous unit, Strategies are at the core of federated learning. They determine how clients are selected, which updates are used, and how the new changes are aggregated.

In this unit, we will focus on custom Strategies. To begin, we need to set up the environment for this notebook's development.

### Exercise
As you are familiar with one of the deep learning frameworks, you can implement the following part based on your preference, either PyTorch, Tensorflow, or JAX.


In [1]:
import numpy as np
import tensorflow as tf
from typing import List, Dict, Optional, Tuple, Union


import flwr as fl
from flwr.common import Context
from flwr.client import ClientApp


#Load the CIFAR-10 in NUM_CLIENTS different subsets for the training and test as it has been in the previous unit

NUM_CLIENTS = 10

def unison_shuffled_copies(a, b):
    assert len(a) == len(b)
    p = np.random.permutation(len(a))
    return a[p], b[p]

def split_index(a, n):
    s = np.array_split(np.arange(len(a)), n)
    return s

# Code to load the dataset
def load_datasets(num_clients: int):
    # Distribute it to train and test set
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    # Normalize data
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    x_train, y_train = x_train[:10_000], y_train[:10_000]
    x_test, y_test = x_test[:1000], y_test[:1000]

    # Randomize the datasets
    x_train, y_train = unison_shuffled_copies(x_train, y_train)
    x_test, y_test = unison_shuffled_copies(x_test, y_test)

    # Split training set into 'num_clients' partitions to simulate the individual dataset
    train_index = split_index(x_train, num_clients)
    test_index = split_index(x_test, num_clients)

    # Split each partition
    train_ds = []
    val_ds = []
    test_ds = []
    for cid in range(num_clients):
        val_size = len(train_index[cid]) // 10
        train_input_data, train_output_data = x_train[train_index[cid]], y_train[train_index[cid]]
        val_input_data, val_output_data = train_input_data[:val_size], train_output_data[:val_size]
        train_input_data, train_output_data = train_input_data[val_size:], train_output_data[val_size:]
        train_dataset = (train_input_data, train_output_data)
        val_dataset = (val_input_data, val_output_data)
        test_dataset = (x_test[test_index[cid]], y_test[test_index[cid]])  
        train_ds.append(train_dataset)
        val_ds.append(val_dataset)
        test_ds.append(test_dataset)
    
    return train_ds, val_ds, test_ds


trainloaders, valloaders, testloader = load_datasets(NUM_CLIENTS)

# Define the model to be used in the clients

# The part to adjust for each framework
def generate_ann():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(),
        metrics=['accuracy']
    )
    return model


def get_parameters(net) -> List[np.array]:
    return net.get_weights()


def set_parameters(net, parameters: List[np.ndarray]):
    net.set_weights(parameters)
    return net


def train(net, trainloader, epochs: int):
    net.fit(trainloader[0], trainloader[1], epochs=epochs, batch_size=32, steps_per_epoch=3)
    return net


def test(net, testloader):
    loss, accuracy = net.evaluate(testloader[0], testloader[1])
    return loss, accuracy

# Class to contain a Client
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, cid, net, trainloader, valloader):
        self.cid = cid
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self, config):
        print(f"[Client {self.cid}] get_parameters")
        return get_parameters(self.net)

    def fit(self, parameters, config):
        print(f"[Client {self.cid}] fit, config: {config}")
        self.net = set_parameters(self.net, parameters)
        self.net = train(self.net, self.trainloader, epochs=1)
        return get_parameters(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        print(f"[Client {self.cid}] evaluate, config: {config}")
        self.net = set_parameters(self.net, parameters)
        loss, accuracy = test(self.net, self.valloader)
        print(f"[Client {self.cid}] loss:{loss}, Client {self.cid} accuracy:{accuracy}")
        return float(loss), len(self.valloader), {"accuracy": float(accuracy)}

def client_fn(Context) -> FlowerClient:
    # Create the model
    net = generate_ann()
    #get the identification
    partition_id = int(Context.node_config["partition-id"])
    #Take the appropiate part of the dataset
    trainloader = trainloaders[int(partition_id)]
    valloader = valloaders[int(partition_id)]
    #Create and return the Client
    return FlowerClient(partition_id, net, trainloader, valloader).to_client()

# def client_fn(cid: str) -> FlowerClient:  # Use str, not Context
#     # Create the model
#     net = generate_ann()
#     #get the identification
#     partition_id = int(cid)  # Directly use cid
#     #Take the appropiate part of the dataset
#     trainloader = trainloaders[partition_id]
#     valloader = valloaders[partition_id]
#     #Create and return the Client
#     return FlowerClient(partition_id, net, trainloader, valloader).to_client()

/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/keras/src/utils/file_utils.py:115: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(


Considering the previous code, the model developed has several possibilities for implementing the Strategy object such as `FedAvg` or `FedAdagrad`,  as seen in the  previous Unit. For example, the following code should create a strategy. 

In [3]:
import flwr as fl
# Create an instance of the model and get the parameters
model = generate_ann()
params = get_parameters(model)

In [4]:
from flwr.server import ServerApp, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.common import ndarrays_to_parameters

num_rounds = 3

def server_fn(context: Context):
    # Generate the model and parameters
    model = generate_ann()
    params = get_parameters(model)
    del model
    global_model_init = ndarrays_to_parameters(params)

    # Create FedAvg strategy
    strategy = FedAvg(
        fraction_fit=0.3,  
        fraction_evaluate=0.3,  
        min_fit_clients=3,
        min_evaluate_clients=2,
        min_available_clients=NUM_CLIENTS, 
        initial_parameters=global_model_init, # Initial parameters
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    # Return the configuration and strategy for this server
    return ServerAppComponents(strategy=strategy, config=config)

# Create Server
server_app = ServerApp(server_fn=server_fn)

# Create Client
client_app = ClientApp(client_fn=client_fn)

#Start the simulation
fl.simulation.run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
)

INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=76644, ip=127.0.0.1, actor_id=c03f1ea306a8402ee053460001000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x112c146e0>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return

(ClientAppActor pid=76644) [Client 4] fit, config: {}


ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=76643, ip=127.0.0.1, actor_id=0d624e1d3756e2c4a8fa6b8b01000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x108414620>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/message_handler/message_handler.py", line 135, in handle_legacy_message_from_msgtype
    evaluate_res = maybe_call_evaluate(
                   ^^^^^^^^^

(ClientAppActor pid=76643) [Client 4] evaluate, config: {}


ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=76644, ip=127.0.0.1, actor_id=c03f1ea306a8402ee053460001000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x112c146e0>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/message_handler/message_handler.py", line 128, in handle_legacy_message_from_msgtype
    fit_res = maybe_call_fit(
              ^^^^^^^^^^^^^^^
  File "

(ClientAppActor pid=76642) [Client 0] fit, config: {} [repeated 8x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(ClientAppActor pid=76642) [Client 3] evaluate, config: {} [repeated 5x across cluster]


It may be worth mentioning that Flower, by default, initializes the global model by making a call to one random client before distributing it to the remaining clients. However, sometimes more control is required, such as when performing fine-tuning. In such situations, we use server-side initialization, and the `initial_parameters` parameter will hold the initial version of the model for all clients. It is important to note that this parameter must be a serialization of the data, so the utility function `ndarrays_to_parameters` can be quite handy in this case.

Now, let's move on to customizing the type of evaluation performed on the models. Broadly speaking, there are two possibilities: server-side evaluation and client-side evaluation.

**Centralized evaluation** (server-side) is similar to traditional machine learning, where the server holds a partition solely for evaluating the aggregated model. This approach reduces communication and is suitable for situations with limited bandwidth. There is no need to send the model to the clients for evaluation, and the entire evaluation dataset is available at all times.

**Federated evaluation** (client-side) is more complex, but it usually represents real-world scenarios more accurately. In this approach, the evaluation dataset is distributed among the clients, which means that we can leverage a larger dataset spread among the resources of the clients. However, this approach comes with a cost. Since we don't have a central dataset, we should be aware that our evaluation dataset can change over consecutive rounds of learning if some clients are not always available. Moreover, the dataset held by each client can also change over consecutive rounds. This can lead to evaluation results that are not stable, so even if we don't change the model, we can see our evaluation results fluctuate over consecutive rounds. Additionally, this approach can significantly increase the number of communications because the models have to be distributed among the clients and retrieved for evaluation.

The previous code snippet is an example of Flower performing Federated evaluations, as it uses the `evaluation` function that is executed on each `Client` and later aggregated after being sent to the server. On the other hand, a Centralized evaluation could be performed with a similar approach, as shown in the following code snippet:


In [5]:
def get_test_loader(input_dataset):
    # The `evaluate` function will be by Flower called after every round
    def evaluate_fn(
        server_round: int, parameters: fl.common.NDArrays, 
        config: Dict[str, fl.common.Scalar]) -> Optional[Tuple[float, Dict[str, fl.common.Scalar]]]:
        # Load the test data
        dataset = input_dataset
        
        # Update the model with the latest parameters
        model = generate_ann()
        set_parameters(model, parameters)
    
        # Evaluate the model on the test dataset
        loss, accuracy = test(model, dataset)
    
        # Log the evaluation results
        print(f"Server-side evaluation round {server_round} with loss {loss} / accuracy {accuracy}")
        
        return loss, {"accuracy": accuracy}

    return evaluate_fn

def server_fn(context: Context):
    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
            fraction_fit=0.3,  
            fraction_evaluate=0.3,  
            min_fit_clients=3,
            min_evaluate_clients=2,
            min_available_clients=NUM_CLIENTS, 
            initial_parameters=fl.common.ndarrays_to_parameters(params),
            evaluate_fn=get_test_loader(testloader[0]),  # Pass the evaluation function
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    # Return the configuration and strategy for this server
    return ServerAppComponents(strategy=strategy, config=config)

# Create Server
server_app = ServerApp(server_fn=server_fn)

# Start the simulation
fl.simulation.run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
)


INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1473 - loss: 2.3791


INFO :      initial parameters (loss, other metrics): 2.357651472091675, {'accuracy': 0.1599999964237213}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 10)


Server-side evaluation round 0 with loss 2.357651472091675 / accuracy 0.1599999964237213
(ClientAppActor pid=55237) [Client 0] fit, config: {}


INFO :      aggregate_fit: received 3 results and 0 failures


...

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0807 - loss: 2.3139   [repeated 10x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0561 - loss: 2.3153


INFO :      fit progress: (3, 2.3082053661346436, {'accuracy': 0.07000000029802322}, 11.501882124997792)
INFO :      configure_evaluate: strategy sampled 3 clients (out of 10)


Server-side evaluation round 3 with loss 2.3082053661346436 / accuracy 0.07000000029802322


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 3 round(s) in 13.09s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.457659641901652
INFO :      		round 2: 2.3008920351664224
INFO :      		round 3: 2.2747925917307534
INFO :      	History (loss, centralized):
INFO :      		round 0: 2.357651472091675
INFO :      		round 1: 2.4364218711853027
INFO :      		round 2: 2.317767858505249
INFO :      		round 3: 2.3082053661346436
INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 0.1599999964237213),
INFO :      	              (1, 0.07000000029802322),
INFO :      	              (2, 0.07000000029802322),
INFO :      	              (3, 0.07000000029802322)]}
INFO :      


(ClientAppActor pid=55237) [Client 5] evaluate, config: {} [repeated 6x across cluster]
(ClientAppActor pid=55238) [Client 4] loss:2.2979178428649902, Client 4 accuracy:0.12999999523162842 [repeated 5x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step - accuracy: 0.0838 - loss: 2.3101
(ClientAppActor pid=55238) [Client 5] fit, config: {} [repeated 2x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1025 - loss: 2.2781   [repeated 5x across cluster]
(ClientAppActor pid=55238) [Client 0] evaluate, config: {} [repeated 2x across cluster]
(ClientAppActor pid=55238) [Client 0] loss:2.288536787033081, Client 0 accuracy:0.10000000149011612 [repeated 3x across cluster]


Additionally, it is possible to implement a custom strategy from scratch by implementing the necessary methods and extending `flwr.server.strategy.Strategy`. The required methods for a custom strategy are as follows:
* `num_fit_clients`: returns the number of clients to be selected for the next round of training.
* `num_rounds`: returns the number of rounds of training to perform.
* `on_fit`: called when a client has completed training and returned its updated model. This method should update the global model based on the returned model.
* `on_evaluate`: called when a client has completed an evaluation and returned its evaluation result. This method should aggregate the evaluation results.


You can see an schema of the methods and an example in the following [link](https://flower.ai/docs/framework/tutorial-series-build-a-strategy-from-scratch-pytorch.html#Build-a-Strategy-from-scratch)

# Challenges for Federated Learning

While federated learning can solve problems that traditional centralized machine learning struggles with, such as privacy and reduced hardware requirements, it also presents its own challenges. In this section, we will cover some of these challenges, including the non-IID (independent and identically distributed) nature of the data, the heterogeneous nature of devices, and the limited communication bandwidth.

## Non-IID data
The assumption of independence and identical distribution, or i.i.d., is commonly made in machine learning and statistical analysis. This means that each data point is independent of all other data points, and that the distribution of the data is the same across all data points.

Non-i.i.d. data, on the other hand, violates one or both of these assumptions. This can occur for a variety of reasons. For example, data may be collected in a way that introduces dependencies between data points, such as when data is collected over time or in a specific order. Additionally, the distribution of the data may vary across different subgroups or regions, making it non-i.i.d. Non-i.i.d. data is a common challenge in federated learning because the data is distributed across many devices, and each device may have a different distribution of data due to variations in data collection methods or data sources. As a result, traditional machine learning algorithms may not perform well on non-i.i.d. data.

In a non-IID data problem (see Figure 1(a)), "non-IIDness" (see Figure 1(c)) refers to the presence of couplings (such as co-occurrence, neighborhood, dependency, linkage, correlation, and causality) and heterogeneities within and between two or more aspects, such as entities, entity classes, entity properties (variables), processes, facts, and states of affairs, or other types of entities or properties (such as learners and learned results) that appear or are produced prior to, during, and after a target process (such as a learning task). Conversely, IIDness ignores or simplifies these relationships, as shown in Figure 1(b).

![Diagram with IID and non-IID data](https://datasciences.org/wp-content/themes/dslabNew/images/datasciences/IIDness.png)
Credit: [Source of the image](https://datasciences.org/non-iid-learning/)

Non-i.i.d. data can be more challenging to work with than i.i.d. data because standard statistical assumptions and techniques may not be applicable. Therefore, special techniques may need to be employed to analyze non-i.i.d. data, which may include techniques that take into account the dependencies between data points or the varying data distributions.

In this context, non-i.i.d. data refers to the fact that the data on each device may differ in terms of distribution, characteristics, and relevance to the task at hand. For instance, the data on one device may comprise mainly images of dogs, while the data on another device may consist mainly of images of cats. This can pose a challenge in training a model that performs well on all the devices because the data on each device can vary significantly from the data on the other devices.

To address non-i.i.d. data in federated learning, special techniques are often employed to weigh the contributions of each device's data to the overall model, or to adjust the model's parameters in a way that considers the differences in the data. Furthermore, techniques such as data augmentation and transfer learning could help to generalize the model beyond the device's data.

When discussing Flower, the approach to addressing this problem would involve [implementing](https://flower.ai/docs/framework/how-to-implement-strategies.html) a custom strategy, similar to the following example, that uses a custom aggregation of the results.


In [6]:
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy
from flwr.server import ServerConfig

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    #aggregate_evaluate is responsible for aggregating the results 
    #returned by the clients that were selected and asked to evaluate in configure_evaluate.
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""

        if not results:
            return None, {}

        # Call aggregate_evaluate from base class (FedAvg) to aggregate loss and metrics
        aggregated_loss, aggregated_metrics = super().aggregate_evaluate(server_round, results, failures)

        # Weigh accuracy of each client by number of examples used
        accuracies = [r.metrics["accuracy"] * r.num_examples for _, r in results]
        examples = [r.num_examples for _, r in results]

        # Aggregate and print custom metric
        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"Round {server_round} accuracy aggregated from client results: {aggregated_accuracy}")

        # Return aggregated loss and metrics (i.e., aggregated accuracy)
        return aggregated_loss, {"accuracy": aggregated_accuracy}


def server_fn(context: Context):
    # instantiate the model
    model = generate_ann()
    params = get_parameters(model)
    del model
    global_model_init = ndarrays_to_parameters(params)

    # Create strategy and run server
    strategy = AggregateCustomMetricStrategy(
        fraction_fit=0.3,
        fraction_evaluate=0.3,
        min_fit_clients=3,
        min_evaluate_clients=2,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=global_model_init,
        evaluate_fn=get_test_loader(testloader[0]),
    )
# Construct ServerConfig
    config = ServerConfig(num_rounds=num_rounds)

    # Wrap everything into a `ServerAppComponents` object
    return ServerAppComponents(strategy=strategy, config=config)


# Create your ServerApp
server_app = ServerApp(server_fn=server_fn)

#Start the simulation
fl.simulation.run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
)


INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0735 - loss: 2.4190


INFO :      initial parameters (loss, other metrics): 2.442880630493164, {'accuracy': 0.09000000357627869}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 10)


Server-side evaluation round 0 with loss 2.442880630493164 / accuracy 0.09000000357627869
(ClientAppActor pid=55526) [Client 9] fit, config: {}


INFO :      aggregate_fit: received 3 results and 0 failures


...

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1055 - loss: 2.3035   [repeated 12x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1169 - loss: 2.3222  


INFO :      fit progress: (3, 2.315891742706299, {'accuracy': 0.10999999940395355}, 11.592884124998818)
INFO :      configure_evaluate: strategy sampled 3 clients (out of 10)


Server-side evaluation round 3 with loss 2.315891742706299 / accuracy 0.10999999940395355
(ClientAppActor pid=55527) [Client 4] evaluate, config: {} [repeated 6x across cluster]
(ClientAppActor pid=55527) [Client 7] loss:2.356982946395874, Client 7 accuracy:0.10999999940395355 [repeated 5x across cluster]


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 3 round(s) in 13.06s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.3112758000691733
INFO :      		round 2: 2.335461457570394
INFO :      		round 3: 2.297494093577067
INFO :      	History (loss, centralized):
INFO :      		round 0: 2.442880630493164
INFO :      		round 1: 2.3633596897125244
INFO :      		round 2: 2.3053061962127686
INFO :      		round 3: 2.315891742706299
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.14333333571751913),
INFO :      	              (2, 0.08999999985098839),
INFO :      	              (3, 0.11999999731779099)]}
INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 0.09000000357627869),
INFO :      	              (1, 0.10000000149011612),
INFO :      	              (2, 0.10000000149011612),
INFO :      	              (3, 0.10999999940395355)]}
INFO :

Round 3 accuracy aggregated from client results: 0.11999999731779099
(ClientAppActor pid=55527) [Client 2] fit, config: {} [repeated 2x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 735us/step - accuracy: 0.1324 - loss: 2.2958 [repeated 5x across cluster]
(ClientAppActor pid=55524) [Client 3] evaluate, config: {} [repeated 2x across cluster]
(ClientAppActor pid=55527) [Client 4] loss:2.294633388519287, Client 4 accuracy:0.11999999731779099 [repeated 3x across cluster]


## Heterogeneity of the devices

The heterogeneity of the devices in the network, which means they may have different hardware and software configurations and may be running different versions of the operating system, is one of the problems of federated learning. This can lead to several problems, including:

* Inefficient communication: Different devices may have varying network speeds and bandwidth, which can make it difficult to transmit model updates between devices in a timely manner.

* Incompatible updates: If different devices are running different versions of the operating systems, they may not be able to exchange model updates due to compatibility issues.

* Data heterogeneity: The data on different devices may differ in terms of quality, quantity, and format, making it challenging to train a model that generalizes well across all devices.



To mitigate the impact of heterogeneous devices in federated learning, researchers are developing techniques such as device-aware aggregation algorithms and communication optimization. These techniques aim to address issues such as inefficient communication and incompatible updates resulting from differences in network speeds, bandwidth, operating system versions, and data heterogeneity across the devices.


Consider a network of five devices (A, B, C, D, and E) that are participating in federated learning to train a global model. Each device has its own data and trains a local model on that data. The local models are then transmitted back to a central server, where they are aggregated and used to update the global model.


In the above scenario, the participating devices (A, B, C, D, and E) in the federated learning network are heterogeneous in nature, meaning they possess different hardware and software configurations. For instance, Device A and Device B may be running distinct versions of the operating system, and Device C may have a slower network connection in comparison to the other devices.


This heterogeneity in the devices can create challenges in the federated learning process. For instance, Device A may face difficulty sending its local model update to the server because of compatibility issues with Device B, and Device C may experience a slower transmission due to its slower network connection.


To overcome the challenges posed by heterogeneous devices in federated learning, researchers are developing techniques to mitigate their impact. These techniques may include device-aware aggregation algorithms, which take into account the different hardware and software configurations of the devices, and communication optimization techniques such as data compression and intelligent routing. By adapting the way that data is aggregated and transmitted, these techniques can help to ensure that all devices are able to contribute effectively to the global model, regardless of their individual characteristics.



It is also worth mentioning that a local configuration can be provided to the `Clients` by means of the `config` parameter of the function in the `FlowerClient`. This parameter is a Python `Dict` which holds values that can be used internally for different purposes, such as limiting the number of epochs on certain clients or establishing the number of rounds.


The modification for the strategy in this case would require the use of parameter `on_fit_config` to indicate the function to retrieve the correct configuration.


```python

...

def fit_config(server_round: int):
    """Return training configuration dict for each round.

    Perform two rounds of training with one local epoch, increase to two local
    epochs afterwards.
    """
    config = {
        "server_round": server_round,  # The current round of federated learning
        "local_epochs": 1 if server_round < 2 else 2,
    }
    return config

...

strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.3,
    fraction_evaluate=0.3,
    min_fit_clients=3,
    min_evaluate_clients=3,
    min_available_clients=10,
    initial_parameters=fl.common.ndarrays_to_parameters(get_parameters(model)),
    evaluate_fn=evaluate,
    on_fit_config_fn=fit_config,  # Pass the fit_config function
)

...
```

However, sometimes limiting the number of rounds or the number of epochs for each client is not enough, especially when the number of clients is too large to handle. In such cases, it may be necessary to reduce the number of clients used for training and evaluation. For instance, consider a scenario where there are 1000 clients, each with only 50 samples for training and 10 for evaluation. Although the amount of data in each client is limited, the communication overhead can still be overwhelming. In such cases, it is better to train for a longer time with a smaller number of clients in each round.

In [7]:
NUM_CLIENTS = 1000

trainloaders, valloaders, testloader = load_datasets(NUM_CLIENTS)

def fit_config(server_round: int):
    config = {
        "server_round": server_round,
        "local_epochs": 3,
    }
    return config

def server_fn(context: Context):
    # instantiate the model
    model = generate_ann()
    params = get_parameters(model)
    del model

    # Create strategy and run server
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=0.025,  # Train on 25 clients (each round)
        fraction_evaluate=0.05,  # Evaluate on 50 clients (each round)
        min_fit_clients=20,
        min_evaluate_clients=40,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
        on_fit_config_fn=fit_config
    )
   
     # Construct ServerConfig
    config = ServerConfig(num_rounds=5)

    # Wrap everything into a `ServerAppComponents` object
    return ServerAppComponents(strategy=strategy, config=config)

# Create your ServerApp
server_app = ServerApp(server_fn=server_fn)

#Start the simulation
fl.simulation.run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
)

/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/keras/src/utils/file_utils.py:115: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(
INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 20 clients (out of 1000)


(ClientAppActor pid=55848) [Client 143] fit, config: {'local_epochs': 3, 'server_round': 1}
(ClientAppActor pid=55851) [Client 237] fit, config: {'server_round': 1, 'local_epochs': 3}


(ClientAppActor pid=55851) 2025-03-13 19:35:33.696168: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
(ClientAppActor pid=55851) 	 [[{{node IteratorGetNext}}]]
(ClientAppActor pid=55851) /Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
(ClientAppActor pid=55851)   self.gen.throw(value)


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1111 - loss: 2.3833  
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 0.0000e+00 - loss: 2.8399
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0000e+00 - loss: 2.8399  
(ClientAppActor pid=55850) [Client 720] fit, config: {'local_epochs': 3, 'server_round': 1} [repeated 5x across cluster]
(ClientAppActor pid=55850) [Client 986] fit, config: {'server_round': 1, 'local_epochs': 3} [repeated 6x across cluster]
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 2.4336   [repeated 10x across cluster]


(raylet) [2025-03-13 19:35:39,008 E 55835 3778015] file_system_monitor.cc:116: /tmp/ray/session_2025-03-13_19-35-28_070906_54432 is over 95% full, available space: 17.1031 GB; capacity: 460.432 GB. Object creation will fail if spilling is required.
(ClientAppActor pid=55850) 2025-03-13 19:35:35.464102: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence [repeated 7x across cluster]
(ClientAppActor pid=55850) 	 [[{{node IteratorGetNext}}]] [repeated 7x across cluster]
(ClientAppActor pid=55850) /Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset. [repeated 3x across cluster]
(ClientAppActor pid=55850)   self.gen.throw(value) [repeated 3x across cluster]


...

(ClientAppActor pid=55849) [Client 143] fit, config: {'local_epochs': 3, 'server_round': 5}


INFO :      aggregate_fit: received 1 results and 24 failures
INFO :      configure_evaluate: strategy sampled 50 clients (out of 1000)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     Traceback (most recent call last):
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/server/superlink/fleet/vce/backend/raybackend.py", line 166, in process_message
    future = self.pool.submit(
             ^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/simulation/ray_

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1111 - loss: 2.1758   [repeated 7x across cluster]


ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     cannot access local variable 'future' where it is not associated with a value
ERROR :     Traceback (most recent call last):
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/server/superlink/fleet/vce/backend/raybackend.py", line 166, in process_message
    future = self.pool.submit(
             ^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/simulation/ray_transport/ray_actor.py", line 458, in submit
    actor = self.pool.pop()
            ^^^^^^^^^^^^^^^
IndexError: pop from empty list

During handling of the above exception, another exception occurred:

Traceback

(ClientAppActor pid=55849) [Client 96] evaluate, config: {} [repeated 3x across cluster]
(ClientAppActor pid=55849) [Client 96] loss:1.4287906885147095, Client 96 accuracy:1.0 [repeated 4x across cluster]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 1.0000 - loss: 1.4288 [repeated 2x across cluster]


In addition to the techniques mentioned earlier, federated transfer learning, secure aggregation, and data augmentation are other approaches that can help in the scaling of the federated learning system. The limitation of resources, including bandwidth, storage, and computation power, is one of the main challenges of federated learning.


### Exercise

As evident from the previous results, the outcomes are not remarkable, mainly attributed to the limited number of patterns for each client. In response, suggest an alternative architecture for the network and experiment with at least four different configurations for the fraction_fit and evaluate. Subsequently, analyze the data and draw conclusions from your findings.

In [2]:
# # Define a new architecture for the neural network
# def generate_new_ann():
#     model = tf.keras.Sequential([
#         tf.keras.layers.Input(shape=(32, 32, 3)),
#         tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
#         tf.keras.layers.MaxPooling2D((2, 2)),
#         tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
#         tf.keras.layers.MaxPooling2D((2, 2)),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dense(128, activation='relu'),
#         tf.keras.layers.Dense(10, activation='softmax')
#     ])

#     model.compile(
#         loss=tf.keras.losses.sparse_categorical_crossentropy,
#         optimizer=tf.keras.optimizers.Adam(),
#         metrics=['accuracy']
#     )
#     return model

# The part to adjust for each framework
def generate_new_ann():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(),
        metrics=['accuracy']
    )
    return model

def get_parameters(model):
    return model.get_weights()

def set_parameters(model, parameters):
    model.set_weights(parameters)

def get_test_loader(test_data):
    def evaluate_fn(server_round, parameters, config):
        model = generate_new_ann()
        set_parameters(model, parameters)
        loss, accuracy = model.evaluate(test_data[0], test_data[1])
        return loss, {"accuracy": accuracy}
    return evaluate_fn

# Experiment with different configurations for fraction_fit and fraction_evaluate
configurations = [
    {"fraction_fit": 0.1, "fraction_evaluate": 0.1},
    {"fraction_fit": 0.2, "fraction_evaluate": 0.2},
    {"fraction_fit": 0.3, "fraction_evaluate": 0.3},
    {"fraction_fit": 0.4, "fraction_evaluate": 0.4},
]

results = []

# Define the number of clients
NUM_CLIENTS = 5
num_rounds = 3

for config in configurations:
    def server_fn(context: Context):
        # Generate the model and parameters
        model = generate_new_ann()
        params = get_parameters(model)
        del model
        global_model_init = ndarrays_to_parameters(params)

        # Create FedAvg strategy
        strategy = FedAvg(
            fraction_fit=config["fraction_fit"],
            fraction_evaluate=config["fraction_evaluate"],
            min_fit_clients=3,
            min_evaluate_clients=2,
            min_available_clients=NUM_CLIENTS,
            initial_parameters=global_model_init,
            evaluate_fn=get_test_loader(testloader[0]),
        )

        # Define ServerConfig
        server_config = fl.server.ServerConfig(num_rounds=num_rounds)

        # Return the configuration and strategy for this server
        return ServerAppComponents(strategy=strategy, config=server_config)

    # Create Server
    server_app = ServerApp(server_fn=server_fn)

    # Create Client
    client_app = ClientApp(client_fn=client_fn)

    # Start the simulation
    history = fl.simulation.run_simulation(
        server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
    )

    results.append({
        "config": config,
        "history": history
    })

# Analyze the results
for result in results:
    config = result["config"]
    history = result["history"]
    print(f"Configuration: {config}")
    print(f"History: {history}")

NameError: name 'ServerApp' is not defined

In [ ]:
# import numpy as np
# import tensorflow as tf
# import flwr as fl
# from flwr.common import Context, ndarrays_to_parameters
# from flwr.server.strategy import FedAvg
# from flwr.server import ServerApp, ServerAppComponents
# from flwr.client import ClientApp

# # # Define a new architecture for the neural network
# # def generate_new_ann():
# #     model = tf.keras.Sequential([
# #         tf.keras.layers.Input(shape=(32, 32, 3)),
# #         tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
# #         tf.keras.layers.MaxPooling2D((2, 2)),
# #         tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
# #         tf.keras.layers.MaxPooling2D((2, 2)),
# #         tf.keras.layers.Flatten(),
# #         tf.keras.layers.Dense(128, activation='relu'),
# #         tf.keras.layers.Dense(10, activation='softmax')
# #     ])
# #     model.compile(
# #         loss=tf.keras.losses.sparse_categorical_crossentropy,
# #         optimizer=tf.keras.optimizers.Adam(),
# #         metrics=['accuracy']
# #     )
# #     return model

# # The part to adjust for each framework
# def generate_new_ann():
#     model = tf.keras.Sequential([
#         tf.keras.layers.Input(shape=(32, 32, 3)),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dense(64, activation='relu'),
#         tf.keras.layers.Dense(64, activation='relu'),
#         tf.keras.layers.Dense(10, activation='softmax')
#     ])
#     model.compile(
#         loss=tf.keras.losses.sparse_categorical_crossentropy,
#         optimizer=tf.keras.optimizers.Adam(),
#         metrics=['accuracy']
#     )
#     return model

# def get_parameters(model):
#     return model.get_weights()

# def set_parameters(model, parameters):
#     model.set_weights(parameters)

# def get_test_loader(test_data):
#     def evaluate_fn(server_round, parameters, config):
#         model = generate_new_ann()
#         set_parameters(model, parameters)
#         loss, accuracy = model.evaluate(test_data[0], test_data[1])
#         return loss, {"accuracy": accuracy}
#     return evaluate_fn

# # Experiment with different configurations for fraction_fit and fraction_evaluate
# configurations = [
#     {"fraction_fit": 0.1, "fraction_evaluate": 0.1},
#     {"fraction_fit": 0.2, "fraction_evaluate": 0.2},
#     {"fraction_fit": 0.3, "fraction_evaluate": 0.3},
#     {"fraction_fit": 0.4, "fraction_evaluate": 0.4},
# ]

# results = []

# # Define the number of clients
# NUM_CLIENTS = 5
# num_rounds = 3

# for config in configurations:
#     def server_fn(context: Context):
#         model = generate_new_ann()
#         params = get_parameters(model)
#         del model
#         global_model_init = ndarrays_to_parameters(params)

#         strategy = FedAvg(
#             fraction_fit=config["fraction_fit"],
#             fraction_evaluate=config["fraction_evaluate"],
#             min_fit_clients=3,
#             min_evaluate_clients=2,
#             min_available_clients=NUM_CLIENTS,
#             initial_parameters=global_model_init,
#             evaluate_fn=get_test_loader(testloader[0]),
#         )

#         server_config = fl.server.ServerConfig(num_rounds=num_rounds)
#         return ServerAppComponents(strategy=strategy, config=server_config)

#     server_app = ServerApp(server_fn=server_fn)
#     client_app = ClientApp(client_fn=client_fn)

#     history = fl.simulation.run_simulation(
#         server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
#     )

#     results.append({
#         "config": config,
#         "history": history
#     })

# for result in results:
#     config = result["config"]
#     history = result["history"]
#     print(f"Configuration: {config}")
#     print(f"History: {history}")


            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.0000e+00 - loss: 2.7180


INFO :      initial parameters (loss, other metrics): 2.7180190086364746, {'accuracy': 0.0}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=56850, ip=127.0.0.1, actor_id=8479f1a72de8ca7b1f76d7b501000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x106622ba0>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^

(ClientAppActor pid=56848) [Client 3] fit, config: {}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.0000e+00 - loss: 2.7180


INFO :      fit progress: (1, 2.7180190086364746, {'accuracy': 0.0}, 5.816775790997781)
INFO :      configure_evaluate: strategy sampled 2 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=56850, ip=127.0.0.1, actor_id=8479f1a72de8ca7b1f76d7b501000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x106622ba0>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/cl

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.0000e+00 - loss: 2.1187


INFO :      fit progress: (2, 2.118725299835205, {'accuracy': 0.0}, 7.528816791993449)
INFO :      configure_evaluate: strategy sampled 2 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=57635, ip=127.0.0.1, actor_id=c5902a83c5e00ba46d2f7ecf01000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x106d14710>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/cli

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.0000e+00 - loss: 2.1187


INFO :      fit progress: (3, 2.118725299835205, {'accuracy': 0.0}, 9.259447499993257)
INFO :      configure_evaluate: strategy sampled 2 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     ray::ClientAppActor.run() (pid=57635, ip=127.0.0.1, actor_id=c5902a83c5e00ba46d2f7ecf01000000, repr=<flwr.simulation.ray_transport.ray_actor.ClientAppActor object at 0x106d14710>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 143, in __call__
    return self._call(message, context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/client/client_app.py", line 126, in ffn
    out_message = handle_legacy_message_from_msgtype(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/flwr/cli

(ClientAppActor pid=57635) [Client 1] fit, config: {} [repeated 8x across cluster]
(ClientAppActor pid=57633) [Client 3] evaluate, config: {} [repeated 5x across cluster]
Configuration: {'fraction_fit': 0.1, 'fraction_evaluate': 0.1}
History: None
Configuration: {'fraction_fit': 0.2, 'fraction_evaluate': 0.2}
History: None
Configuration: {'fraction_fit': 0.3, 'fraction_evaluate': 0.3}
History: None
Configuration: {'fraction_fit': 0.4, 'fraction_evaluate': 0.4}
History: None
